# Solving Differential Equations Numerically

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/DiffEQTutorial.m`](../matlab/DiffEQTutorial.m) by Fred Rieke.

Most quantitative models in neuroscience are differential equations. A membrane time
constant, a synaptic conductance, a second-messenger cascade, an ion-channel gating
particle — all of them are statements about how fast something changes as a function of
its current state. Very few of these equations have solutions you can write down. So the
practical skill is turning a differential equation into something a computer can step
forward in time, and knowing when to trust the answer.

That is what this tutorial is about. The central move is simple enough to do by hand:
replace the derivative

$$\frac{dx}{dt} \;\approx\; \frac{x(n) - x(n-1)}{\Delta t}$$

and rearrange into an **update rule** that gives you $x$ in one time bin from $x$ in the
previous bin. Iterating that rule is the **forward Euler method**. We write it out
explicitly — as a `for` loop over time — because that loop *is* the lesson. Once you have
seen it, `scipy.integrate.solve_ivp` stops being a black box.

| Part | Topic |
|---|---|
| I | Building the update rule: creation and decay of a substance |
| II | Adding spontaneous activation |
| III | Adding feedback — where analytical solutions run out |
| IV | Cascades: rhodopsin → phosphodiesterase |
| V | A full phototransduction model: cGMP and membrane current |
| VI | Calcium feedback, and where oscillations come from |
| VII | Two-state systems: Hodgkin–Huxley gating particles |
| VIII | Solving differential equations with Fourier transforms |
| IX | Accuracy and stability: how wrong is Euler, and when does it explode? |
| X | Coupled systems: an oculomotor plant model with `solve_ivp` |

Run it cell by cell (**Shift+Enter**). The homework questions are part of the tutorial,
not decoration — most of them are answered by changing one number and re-running a cell.

**Prerequisites:** basic calculus, and (for Part VIII) the Fourier transform tutorial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "font.size": 10,
    "lines.linewidth": 1.6,
})

# There is no randomness anywhere in this tutorial -- every result below is
# fully deterministic -- but we fix a seed anyway so that any exploration you
# add (noisy inputs, stochastic gating) reproduces run to run.
rng = np.random.default_rng(545)

### A note on translating MATLAB loops

The MATLAB original grows its arrays inside the loop (`x(pnt) = ...` with `pnt` running
from `2` to `NumPts`). That works in MATLAB but is slow, and it hides a bug that bites
everyone once: the array is left over from a previous cell if you forget `clear all`.
In Python we **preallocate** with `np.zeros(NumPts)` and set the initial condition
explicitly. Four differences to keep straight:

| | MATLAB | Python |
|---|---|---|
| First element | `x(1)` | `x[0]` |
| Loop over updates | `for pnt = 2:NumPts` | `for n in range(1, NumPts)` |
| Last element | `x(end)` | `x[-1]` |
| First $N/2$ elements | `x(1:NumPts/2)` | `x[:NumPts//2]` (exclusive end, integer `//`) |

Note especially the time axis. MATLAB writes `tme = (1:NumPts - PrePts) * TimeStep`,
so the *first* sample sits at $(1-\text{PrePts})\Delta t$. In Python
`np.arange(NumPts)` starts at 0, so we use `(np.arange(NumPts) + 1 - PrePts) * TimeStep`
to keep the same alignment: time zero is the last pre-stimulus point.

---
## Part I. Creation and decay: building the update rule

Consider a substance $x$ created by another substance $y$. For example, $y$ could be an
active receptor and $x$ its downstream effector. The rate of creation of $x$ is
proportional to the amount of active $y$, and $x$ decays with rate constant $\alpha$:

$$\frac{dx}{dt} = y(t) - \alpha x.$$

This equation does **not** have a unique solution until we also specify an initial
condition — a point that applies equally to numerical and analytical solutions. We take
$x(0) = 0$.

To solve it numerically, discretize time into steps of length $\Delta t$ and approximate
the derivative by a finite difference. This amounts to a first-order Taylor expansion of
$x(t)$: we assume that over one time step the behavior of $x$ is determined entirely by
its first derivative, ignoring the second derivative and higher terms. So

$$\frac{dx}{dt} \;\longrightarrow\; \frac{x(n) - x(n-1)}{\Delta t},$$

where $x(n)$ is the value of $x$ in the $n$-th time bin, i.e. at time $n\,\Delta t$. In
the limit $\Delta t \to 0$ this *is* the definition of the derivative — make sure you see
that connection, because it is at the core of how differential equations are solved
numerically. The original equation becomes

$$\frac{x(n) - x(n-1)}{\Delta t} = y(n-1) - \alpha\, x(n-1),$$

and solving for $x(n)$ gives the update rule:

$$\boxed{\,x(n) = x(n-1) + \Delta t \left[\, y(n-1) - \alpha\, x(n-1) \,\right]\,}$$

Given $x$ and $y$ in bin $n-1$ we can compute $x$ in bin $n$. Iterating gives an
approximation to the full time course of $x$. The accuracy improves as $\Delta t$ shrinks
— Part IX makes that statement quantitative.

We drive it with a step in $y$: off, then on, then off again.

In [ ]:
# --- Parameters (names kept recognizable against the MATLAB original) ---
alpha = 20.0                # rate constant for decay of x, 1/sec
k = 0.1                     # scale factor: k sets the time step, holding total duration fixed
TimeStep = 0.001 / k        # time step for the difference equation, sec
PrePts   = int(200 * k)     # points before the step in y
StmPts   = int(400 * k)     # points for which y is active
NumPts   = int(1000 * k)    # total points to simulate

# initialize y: a simple step
y = np.zeros(NumPts)
y[PrePts:PrePts + StmPts] = 1.0

# time axis: t = 0 is the last pre-step point (matches the MATLAB alignment)
tme = (np.arange(NumPts) + 1 - PrePts) * TimeStep

# --- The Euler loop. This is the whole method. ---
# Preallocate (MATLAB's original grew the array inside the loop).
x = np.zeros(NumPts)
x[0] = 0.0                  # initial condition
for pnt in range(1, NumPts):
    x[pnt] = x[pnt-1] + (y[pnt-1] - alpha * x[pnt-1]) * TimeStep

# --- Analytical solution, for comparison ---
# While y = 1 (from t = 0 to t = StmPts*dt) the solution rising from x = 0 is
#   x(t) = (1/alpha) * (1 - exp(-alpha t)),
# and after the step turns off it decays as x(t) = x_off * exp(-alpha (t - t_off)).
t_on  = StmPts * TimeStep
x_an  = np.zeros(NumPts)
rise  = (tme >= 0) & (tme < t_on)
fall  = tme >= t_on
x_an[rise] = (1/alpha) * (1 - np.exp(-alpha * tme[rise]))
x_off = (1/alpha) * (1 - np.exp(-alpha * t_on))
x_an[fall] = x_off * np.exp(-alpha * (tme[fall] - t_on))

fig, axes = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
axes[0].plot(tme, y, color="#333333")
axes[0].set(ylim=(-0.1, 1.1), ylabel="input $y$",
            title="activation of $x$ by $y$, no feedback")
axes[1].plot(tme, x, color="#2f6fb5", label=f"Euler, $\\Delta t$ = {TimeStep*1000:.0f} ms")
axes[1].plot(tme, x_an, "--", color="#cc5544", lw=1.3, label="analytical")
axes[1].set(xlabel="time (sec)", ylabel="output $x$")
axes[1].legend(frameon=False)
fig.tight_layout()

print(f"TimeStep = {TimeStep*1000:.1f} ms,  alpha*TimeStep = {alpha*TimeStep:.3f}")
print(f"steady-state x while y = 1:  numerical {x[PrePts+StmPts-1]:.5f}   "
      f"analytical {1/alpha:.5f}")
print(f"max |Euler - analytical| = {np.max(np.abs(x - x_an)):.5f} "
      f"({100*np.max(np.abs(x - x_an))/x_an.max():.1f}% of peak)")

This is an important example because it is one you use all the time. The analytical
solution is a rising and then decaying **exponential**, with time constant
$\tau = 1/\alpha$ and steady-state value $y/\alpha$. Many processes you will meet are
approximated as single exponentials — the membrane time constant of a cell being the most
familiar.

Notice the numerical solution *lags* the analytical one during the rise. That is not a
coding error: forward Euler evaluates the derivative at the *start* of each step and holds
it constant across the step, so for a decaying process it systematically underestimates
how much decay has already happened. The error is first order in $\Delta t$, which we
verify directly in Part IX.

$\alpha \Delta t = 0.2$ here. Keep that number in mind — it is the quantity that decides
whether Euler is accurate, and, past $\alpha \Delta t = 2$, whether it is stable at all.

> ### Homework question 1
> **(a)** Play with the rate constant $\alpha$ and explain what happens to both the
> amplitude and the kinetics of $x$. Which one does $\alpha$ control, and why does the
> other change too?
>
> **(b)** Try differently shaped inputs (a brief pulse, a ramp, a sinusoid) and explain
> the results.
>
> **(c)** Play with `TimeStep` (change `k`) to see over what range of time bins the
> numerical solution is accurate — compare against the analytical solution plotted above,
> using the printed maximum error. What happens once $\alpha \Delta t$ exceeds 2?

---
## Part II. Adding spontaneous activation

Let's elaborate. What if $x$ also has a rate of spontaneous activation, in addition to
activation by $y$? The equation becomes

$$\frac{dx}{dt} = y(t) + \eta - \alpha x,$$

where $\eta$ is the rate of spontaneous activation. The update rule follows exactly as
before, with $\eta$ carried along:

$$x(n) = x(n-1) + \Delta t \left[\, y(n-1) + \eta - \alpha\, x(n-1) \,\right].$$

The interesting question is what initial condition to use. If we want the system to be at
rest before the step arrives, we need $dx/dt = 0$ with $y = 0$, which gives
$x(0) = \eta/\alpha$. Starting anywhere else produces a transient at the beginning of the
simulation that has nothing to do with the stimulus — a very common and very confusing
artifact in your own models.

In [ ]:
eta = 1.0                       # rate of spontaneous activation of x
x2 = np.zeros(NumPts)
x2[0] = eta / alpha             # steady-state initial condition: dx/dt = 0 when y = 0

for pnt in range(1, NumPts):
    x2[pnt] = x2[pnt-1] + (y[pnt-1] + eta - alpha * x2[pnt-1]) * TimeStep

# a deliberately wrong initial condition, to show the start-up transient
x2_bad = np.zeros(NumPts)
x2_bad[0] = 0.0
for pnt in range(1, NumPts):
    x2_bad[pnt] = x2_bad[pnt-1] + (y[pnt-1] + eta - alpha * x2_bad[pnt-1]) * TimeStep

fig, axes = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
axes[0].plot(tme, y, color="#333333")
axes[0].set(ylim=(-0.1, 1.1), ylabel="input $y$",
            title="activation of $x$ by $y$, with spontaneous activation $\\eta$")
axes[1].plot(tme, x2, color="#2f6fb5", label=r"$x(0)=\eta/\alpha$ (at rest)")
axes[1].plot(tme, x2_bad, ":", color="#999999", label=r"$x(0)=0$ (start-up transient)")
axes[1].axhline(eta/alpha, color="#cc5544", lw=1, ls="--")
axes[1].set(xlabel="time (sec)", ylabel="output $x$")
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()

print(f"baseline eta/alpha       = {eta/alpha:.4f}")
print(f"plateau while y = 1      = {x2[PrePts+StmPts-1]:.4f}   "
      f"(analytical (1+eta)/alpha = {(1+eta)/alpha:.4f})")
print(f"step-evoked change       = {x2[PrePts+StmPts-1] - eta/alpha:.4f}   "
      f"(same as Part I: {x[PrePts+StmPts-1]:.4f})")

The step-evoked *change* is identical to Part I — printed above, both about 0.05. Adding
$\eta$ shifts the whole trajectory up by a constant $\eta/\alpha$ but does not change its
shape at all. That is a direct consequence of the equation being **linear**: a constant
input adds a constant offset, and superposition holds. Part III breaks exactly this
property.

> ### Homework question 2
> **(a)** How does the solution differ in this case from that in Part I? Why?
>
> **(b)** Can you explain the change directly from the differential equation, without
> running it?
>
> **(c)** Why is $\eta/\alpha$ a reasonable initial condition? What does the grey dotted
> trace above show you about picking a bad one?

---
## Part III. Adding feedback

Both examples above can be solved analytically, which gave us a useful check on the
numerical solution. Now let's add a **feedback** term, which makes the analytical solution
difficult or impossible — and this is exactly when numerical solutions earn their keep.

Active $x$ feeds back to modify the *effective* activity of $y$, i.e. the rate at which
$y$ creates $x$:

$$\frac{dx}{dt} = y(t)\,\bigl(1 + g x\bigr)^{n} - \alpha x,$$

where $g$ is the gain of the feedback and the exponent $n$ determines how linear or
nonlinear the feedback is. Negative $n$ makes the feedback **negative** (more $x$ suppresses
its own production); positive $n$ makes it **positive**. There are certainly other ways a
feedback mechanism could work, with correspondingly different differential equations, but
this is one reasonable form.

Because the right-hand side now depends on $x$ nonlinearly, superposition fails — you can
no longer scale the input and scale the answer.

**A note on the original code.** The MATLAB version writes `y(pnt)` (the *current* bin)
in this loop, while every other loop in the file uses `y(pnt-1)`. That is a small
inconsistency, not a bug with consequences here — it merely shifts the input by one bin —
but it is worth noticing, because mixing the two conventions in a coupled system is a
genuine source of error. We use `y[pnt-1]` throughout for consistency.

In [ ]:
Power = -4.0                # feedback exponent (negative = negative feedback)
g     = 20.0                # feedback gain
alpha_fb = 10.0

k        = 0.05
TimeStep_fb = 0.001 / k
PrePts_fb   = int(200 * k)
StmPts_fb   = int(400 * k)
NumPts_fb   = int(1000 * k)

y_fb = np.zeros(NumPts_fb)
y_fb[PrePts_fb:PrePts_fb + StmPts_fb] = 1.0
tme_fb = (np.arange(NumPts_fb) + 1 - PrePts_fb) * TimeStep_fb

def solve_feedback(power, gain, alpha_=alpha_fb):
    '''Euler solve of dx/dt = y (1 + g x)^n - alpha x.

    Positive feedback with a large exponent can run away; we bail out rather
    than letting the loop overflow to inf/nan (MATLAB would print Inf and keep
    going, which is easy to miss).
    '''
    xf = np.zeros(NumPts_fb)
    for pnt in range(1, NumPts_fb):
        xf[pnt] = xf[pnt-1] + (y_fb[pnt-1] * (1 + gain * xf[pnt-1])**power
                               - alpha_ * xf[pnt-1]) * TimeStep_fb
        if not np.isfinite(xf[pnt]) or abs(xf[pnt]) > 1e6:
            xf[pnt:] = np.nan
            return xf, False          # diverged
    return xf, True

x_nofb, _ = solve_feedback(0.0, 0.0)       # power 0 => (1+gx)^0 = 1, i.e. no feedback
x_fb, _   = solve_feedback(Power, g)

fig, axes = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
axes[0].plot(tme_fb, y_fb, color="#333333")
axes[0].set(ylim=(-0.1, 1.1), ylabel="input $y$",
            title="activation of $x$ by $y$, with feedback")
axes[1].plot(tme_fb, x_nofb, color="#999999", label="no feedback ($n=0$)")
axes[1].plot(tme_fb, x_fb, color="#2f6fb5", label=f"feedback ($n$={Power:.0f}, $g$={g:.0f})")
axes[1].set(xlabel="time (sec)", ylabel="output $x$")
axes[1].legend(frameon=False)
fig.tight_layout()

print(f"peak without feedback = {x_nofb.max():.5f}")
print(f"peak with feedback    = {x_fb.max():.5f}   "
      f"({100*x_fb.max()/x_nofb.max():.1f}% of the no-feedback peak)")

Negative feedback does two things at once, and it is worth separating them. It **reduces
the amplitude**, because as $x$ builds up it suppresses its own production. And it
**speeds the kinetics** on the rising phase, because the system reaches its (lower)
plateau sooner. The decay after the step is *not* sped up, though: once $y = 0$ the
feedback term is multiplied by zero and $x$ simply decays at rate $\alpha$. Look at the
two traces after the step turns off — they fall in parallel.

Let's sweep the exponent and the gain to see this systematically. The first two panels
share identical axis limits so the comparison between them is honest. Positive feedback
needs its own panel and a logarithmic axis, because it does something the negative-feedback
cases never do.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))

# --- (a) negative feedback: varying the exponent ---
for pw, col in zip([0, -1, -2, -4, -8], plt.cm.viridis(np.linspace(0.15, 0.85, 5))):
    xf_, _ = solve_feedback(pw, g)
    axes[0].plot(tme_fb, xf_, color=col, label=f"$n$ = {pw}")
axes[0].set(xlabel="time (sec)", ylabel="output $x$",
            title=f"exponent $n \\leq 0$   ($g$ = {g:.0f})")
axes[0].legend(frameon=False, fontsize=9)

# --- (b) negative feedback: varying the gain ---
for gain, col in zip([0, 2, 5, 20, 100], plt.cm.viridis(np.linspace(0.15, 0.85, 5))):
    xf_, _ = solve_feedback(Power, gain)
    axes[1].plot(tme_fb, xf_, color=col, label=f"$g$ = {gain}")
axes[1].set(xlabel="time (sec)", title=f"gain $g$   ($n$ = {Power:.0f})")
axes[1].legend(frameon=False, fontsize=9)

# panels (a) and (b) are meant to be compared -> identical limits
for ax in axes[:2]:
    ax.set_xlim(tme_fb[0], tme_fb[-1])
    ax.set_ylim(-0.005, 0.105)

# --- (c) positive feedback, on a log axis: runaway ---
for pw, col in zip([0, 0.5, 1.0, 1.5, 2.0], plt.cm.plasma(np.linspace(0.1, 0.8, 5))):
    xf_, ok = solve_feedback(pw, g)
    axes[2].semilogy(tme_fb, np.where(xf_ > 0, xf_, np.nan), color=col,
                     label=f"$n$ = {pw:g}" + ("" if ok else "  (diverges)"))
axes[2].set(xlabel="time (sec)", ylabel="output $x$ (log scale)",
            xlim=(tme_fb[0], tme_fb[-1]), ylim=(1e-3, 1e3),
            title=f"exponent $n > 0$: positive feedback   ($g$ = {g:.0f})")
axes[2].legend(frameon=False, fontsize=9)
fig.tight_layout()

print("peak x as a function of exponent n (gain g = 20):")
for pw in [2, 1, 0, -2, -4, -8]:
    xf_, ok = solve_feedback(pw, g)
    print(f"   n = {pw:>3} :  " + (f"peak = {np.nanmax(xf_):.5f}" if ok
                                   else "DIVERGED (positive feedback runs away)"))
print("\npeak x as a function of gain g (exponent n = -4):")
for gain in [0, 2, 5, 20, 100]:
    xf_, _ = solve_feedback(Power, gain)
    print(f"   g = {gain:>3} :  peak = {np.nanmax(xf_):.5f}")

Negative exponents shrink the peak, and increasing $|n|$ shrinks it further -- but with
diminishing returns, because once the feedback is strong enough it clamps $x$ near the
value at which production and decay balance. Increasing $g$ does the same thing but for a
different reason: $g$ sets the *scale* of $x$ at which feedback starts to bite (it is
negligible while $gx \ll 1$), while $n$ sets how sharply it engages once it does. They are
not interchangeable.

Positive exponents give **positive feedback**: $x$ boosts its own production. Panel (c) is
on a log axis because the behavior there is qualitatively different -- at $n=1$ the peak is
tens of times the no-feedback peak, and at $n=2$ the loop gain exceeds one and the solution
**runs away**, overflowing before the input step even ends. Our solver detects this and
bails out; the MATLAB original would fill the array with `Inf` and keep going, which is
very easy to miss inside a `for` loop.

That runaway is also a warning about time steps. In a nonlinear system the effective rate
constant depends on the state, so a step size that is comfortably stable at small $x$ can
become unstable once $x$ grows. Stability is a property of the trajectory, not just of the
parameters -- which is exactly the argument for adaptive step-size solvers (Part IX).


> ### Homework question 3
> **(a)** How does the behavior compare to the case without feedback? Compare both the
> amplitude and the kinetics of $x$.
>
> **(b)** How does the effect of feedback change as the power is changed? Why? Try both
> negative and positive values.
>
> **(c)** How does changing $g$ change things? Why?
>
> **(d)** What happens when you change `TimeStep`? Is the numerical solution more or less
> sensitive to `TimeStep` than the one in Part I? Why?

---
## Part IV. Cascades: rhodopsin activates phosphodiesterase

Now let's consider combinations of a couple of steps. Anticipating a model for
phototransduction, we rename the generic $x$ and $y$ to $r$ (rhodopsin activity) and $p$
(phosphodiesterase activity).

Light produces an electrical signal in a photoreceptor by activating rhodopsin, which
then activates phosphodiesterase (PDE) through the G protein transducin. Active PDE breaks
down cGMP, which reduces the membrane current through the cGMP-gated channels.

Start with PDE activation. Assume rhodopsin's activity — meaning its ability to activate
PDE through transducin — decays exponentially:

$$\frac{dr}{dt} = -\sigma r,$$

where $\sigma$ is the decay rate constant. Then PDE activity obeys

$$\frac{dp}{dt} = r + \eta - \phi\, p,$$

where $\eta$ represents spontaneous PDE activation and $\phi$ is the PDE decay rate
constant. This is exactly the Part II equation, driven by $r$ instead of a step.

The initial conditions: $r(0) = 1$ (one rhodopsin molecule activated at $t=0$), and
$p(0) = \eta/\phi$, the dark steady state.

In [ ]:
sigma = 5.0             # rhodopsin activity decay rate constant, 1/sec
phi   = 20.0            # phosphodiesterase decay rate constant, 1/sec
eta   = 10.0            # spontaneous PDE activation rate, 1/sec

NumPtsP  = 1000
TimeStepP = 0.001
tmeP = (np.arange(NumPtsP) + 1) * TimeStepP

def cascade(sigma_=sigma, phi_=phi, eta_=eta, n=NumPtsP, dt=TimeStepP):
    '''Two-stage cascade: rhodopsin decay driving PDE activation.'''
    r = np.zeros(n); p = np.zeros(n)
    r[0] = 1.0                  # one active rhodopsin at t = 0
    p[0] = eta_ / phi_          # dark steady state
    for pnt in range(1, n):
        r[pnt] = r[pnt-1] + dt * (-sigma_ * r[pnt-1])
        p[pnt] = p[pnt-1] + dt * (r[pnt-1] + eta_ - phi_ * p[pnt-1])
    return r, p

r, p = cascade()

fig, axes = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
axes[0].plot(tmeP, r, color="#2f6fb5")
axes[0].plot(tmeP, np.exp(-sigma * tmeP), "--", color="#cc5544", lw=1.2,
             label=r"analytical $e^{-\sigma t}$")
axes[0].set(ylabel="rhodopsin activity $r$", title="activation cascade: $r \\to p$")
axes[0].legend(frameon=False)
axes[1].plot(tmeP, p, color="#2f6fb5")
axes[1].axhline(eta/phi, color="#cc5544", ls="--", lw=1, label=r"dark level $\eta/\phi$")
axes[1].set(xlabel="time (sec)", ylabel="pde activity $p$")
axes[1].legend(frameon=False)
fig.tight_layout()

print(f"rhodopsin time constant 1/sigma = {1/sigma*1000:.0f} ms")
print(f"pde       time constant 1/phi   = {1/phi*1000:.0f} ms")
print(f"dark pde level eta/phi          = {eta/phi:.3f}")
print(f"peak pde                        = {p.max():.4f} at t = {tmeP[p.argmax()]*1000:.0f} ms")
print(f"pde change above dark           = {p.max() - eta/phi:.4f}")

The PDE response peaks well after the rhodopsin activity has begun to fall — printed
above at about 100 ms — because $p$ integrates $r$. A cascade of first-order stages
introduces delay: each stage low-pass filters its input, and the peak of the output slides
later. This is why the photoreceptor's response to a brief flash peaks a couple of hundred
milliseconds after the flash rather than instantly.

Which stage sets the kinetics of $p$? Whichever is **slower**. Here $1/\sigma = 200$ ms
and $1/\phi = 50$ ms, so rhodopsin's decay dominates the falling phase of $p$ and PDE's
own decay controls only the rise. In fact, swapping the two rate constants leaves the
waveform **exactly** unchanged (check the printed peaks and peak times): the cascade is
mathematically symmetric in $\sigma$ and $\phi$. This is precisely why kinetics alone
cannot tell you which molecular step is rate-limiting — a fact worth remembering the next
time you see a paper infer a mechanism from a response time course.

Both panels below plot $p$ with the dark level $\eta/\phi$ subtracted, since changing
$\phi$ also changes where the trace sits at rest; without that subtraction you would be
comparing offsets rather than kinetics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# Both panels show p with the dark level subtracted: changing phi also changes where
# the trace sits at rest, so without the subtraction we would be comparing offsets
# rather than kinetics -- and the two panels could not share a y axis.
for s_, col in zip([2, 5, 20, 50], plt.cm.viridis(np.linspace(0.15, 0.85, 4))):
    _, p_ = cascade(sigma_=s_)
    axes[0].plot(tmeP, p_ - eta/phi, color=col, label=rf"$\sigma$ = {s_}")
axes[0].set(xlabel="time (sec)", ylabel="pde activity above dark",
            title=rf"varying $\sigma$   ($\phi$ = {phi:.0f})")
axes[0].legend(frameon=False, fontsize=9)

for ph_, col in zip([2, 5, 20, 50], plt.cm.viridis(np.linspace(0.15, 0.85, 4))):
    _, p_ = cascade(phi_=ph_, eta_=eta)
    axes[1].plot(tmeP, p_ - eta/ph_, color=col, label=rf"$\phi$ = {ph_}")
axes[1].set(xlabel="time (sec)", title=rf"varying $\phi$   ($\sigma$ = {sigma:.0f})")
axes[1].legend(frameon=False, fontsize=9)
for ax in axes:                     # identical limits -- the panels are meant to be compared
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.005, 0.115)
fig.tight_layout()

print("swapping sigma and phi leaves the pde waveform unchanged:")
_, pA = cascade(sigma_=5.0, phi_=20.0, eta_=0.0)
_, pB = cascade(sigma_=20.0, phi_=5.0, eta_=0.0)
print(f"   sigma=5,  phi=20 : peak {pA.max():.5f} at {tmeP[pA.argmax()]*1000:.0f} ms")
print(f"   sigma=20, phi=5  : peak {pB.max():.5f} at {tmeP[pB.argmax()]*1000:.0f} ms")

> ### Homework question 4
> **(a)** Why do we choose an exponential decay for the shape of rhodopsin's activity?
> What might change that? (Hint: what does the exponential assume about how rhodopsin is
> shut off — one step, or many?)
>
> **(b)** Explore different combinations of $\sigma$ and $\phi$ and their impact. Can you
> tell from the shape of $p(t)$ alone which of the two is rate-limiting?

---
## Part V. A phototransduction model: cGMP and membrane current

Part IV describes the *activation* arm of phototransduction — how light activation of
rhodopsin leads to activation of PDE. Now let's add the steps linking that to a change in
current.

The role of activated PDE is to hydrolyze cGMP. Another enzyme, guanylate cyclase,
synthesizes cGMP. So the cGMP concentration $g$ is set by a balance of synthesis (at rate
$s$) and hydrolysis (at a rate proportional to both the PDE activity and the available
cGMP):

$$\frac{dg}{dt} = s - p\,g.$$

The membrane current depends on the **third power** of the cGMP concentration, because
opening a channel requires binding of about three cGMP molecules:

$$I = k_{\text{cg}}\, g^{3}.$$

This is all we need for a simple phototransduction model. The parameters are pinned down
by the dark steady state: with $g = g_{\text{dark}} = 15$ and $p = \eta/\phi$, requiring
$dg/dt = 0$ gives $s = g_{\text{dark}}\,\eta/\phi$.

In [ ]:
gdark    = 15.0         # dark cGMP concentration
cgmp2cur = 8e-3         # constant relating cGMP^3 to current

g_ = np.zeros(NumPtsP)
s_ = np.full(NumPtsP, gdark * eta / phi)   # constant synthesis: steady state, synthesis = hydrolysis
r_ = np.zeros(NumPtsP); p_ = np.zeros(NumPtsP)
g_[0] = gdark
r_[0] = 1.0
p_[0] = eta / phi

for pnt in range(1, NumPtsP):
    r_[pnt] = r_[pnt-1] + TimeStepP * (-sigma * r_[pnt-1])
    p_[pnt] = p_[pnt-1] + TimeStepP * (r_[pnt-1] + eta - phi * p_[pnt-1])
    g_[pnt] = g_[pnt-1] + TimeStepP * (s_[pnt-1] - p_[pnt-1] * g_[pnt-1])

cur = cgmp2cur * g_**3

fig, axes = plt.subplots(4, 1, figsize=(7.5, 9), sharex=True)
for ax, dat, lab, col in [
        (axes[0], cur, "current (pA)",       "#2f6fb5"),
        (axes[1], p_,  "pde activity",       "#7a5195"),
        (axes[2], s_,  "synthesis rate",     "#bc5090"),
        (axes[3], g_,  "[cGMP]",             "#ef8354")]:
    ax.plot(tmeP, dat, color=col)
    ax.set_ylabel(lab)
axes[2].set_ylim(0, 2 * s_[0])          # a flat line needs a scale, or it looks like noise
axes[0].set_title("phototransduction, no calcium feedback")
axes[-1].set_xlabel("time (sec)")
fig.tight_layout()

print(f"dark current        = {cur[0]:.3f}")
print(f"minimum current     = {cur.min():.3f}  ({100*(1-cur.min()/cur[0]):.2f}% suppression)")
print(f"current at t = 1 s  = {cur[-1]:.3f}  (recovered to "
      f"{100*cur[-1]/cur[0]:.2f}% of dark)")
print(f"steady-state check: s - p*g at t=0 is {s_[0] - p_[0]*g_[0]:.2e}")

Two things to notice. First, the synthesis rate is a flat line — that is the *point* of
this version of the model, and the next section changes it. We gave that panel explicit
limits so you can see it is genuinely constant rather than wandering; the MATLAB original
lets it autoscale, which turns floating-point dust into what looks like a signal.

Second, look at how the response recovers. The current dips by only about 2% (this is a
single-photon-scale response) and then comes back toward the dark level smoothly and
**monotonically** — it is at 98% of dark at 1 s and still creeping up. There is no
overshoot, no undershoot, no ringing, because nothing in this model opposes the change.
Real photoreceptor flash responses undershoot slightly and recover faster than this. That
discrepancy is the motivation for the next section.

---
## Part VI. Calcium feedback

Now let's add a calcium feedback to the model above.

Calcium enters the outer segment through the cGMP-gated channels, so the rate of calcium
influx is proportional to the current flowing. Calcium is removed by a Na$^+$/K$^+$,Ca$^{2+}$
exchanger, at a rate proportional to the calcium concentration:

$$\frac{dc}{dt} = q\,I - \beta c,$$

where $c$ is the calcium concentration, $q$ is the proportionality constant between the
current and calcium influx, and $\beta$ is the rate constant for calcium removal.

Calcium acts on the rate of cGMP synthesis through a Hill relation:

$$s = \frac{s_{\max}}{1 + \left(c / K\right)^{h}},$$

where $s_{\max}$ is the maximum rate and $K$ and $h$ are the affinity and cooperativity
of the feedback. Otherwise the model is the same.

This is a genuine **negative feedback loop**: light $\to$ PDE up $\to$ cGMP down $\to$
current down $\to$ calcium down $\to$ synthesis up $\to$ cGMP back up. Note that the loop
runs through the calcium dynamics, so it acts with a **delay** set by $1/\beta$. Delayed
negative feedback is the standard recipe for oscillation, and we will see exactly that.

As in Part V, $q$ and $s_{\max}$ are not free parameters — they are fixed by requiring
that the dark state be a steady state:

$$q = \frac{\beta\, c_{\text{dark}}}{k_{\text{cg}}\, g_{\text{dark}}^{3}}, \qquad
s_{\max} = \frac{\eta}{\phi}\, g_{\text{dark}} \left[1 + \left(\frac{c_{\text{dark}}}{K}\right)^{h}\right].$$

In [ ]:
cdark        = 0.5      # dark calcium concentration
beta_ca      = 0.1      # rate constant for calcium removal, 1/sec  (original value)
hillcoef     = 4.0      # cooperativity h
hillaffinity = 0.3      # affinity K

def phototransduction(beta_=beta_ca, r0=1.0, n=NumPtsP, dt=TimeStepP,
                      feedback=True):
    '''Full phototransduction model. feedback=False freezes the synthesis rate.'''
    # constants pinned by the dark steady state
    cur2ca = beta_ * cdark / (cgmp2cur * gdark**3)
    smax   = eta/phi * gdark * (1 + (cdark / hillaffinity)**hillcoef)

    r = np.zeros(n); p = np.zeros(n); c = np.zeros(n); s = np.zeros(n); gg = np.zeros(n)
    r[0]  = r0
    p[0]  = eta / phi
    c[0]  = cdark
    gg[0] = gdark
    s[0]  = gdark * eta / phi
    for pnt in range(1, n):
        r[pnt]  = r[pnt-1]  + dt * (-sigma * r[pnt-1])
        p[pnt]  = p[pnt-1]  + dt * (r[pnt-1] + eta - phi * p[pnt-1])
        c[pnt]  = c[pnt-1]  + dt * (cur2ca * cgmp2cur * gg[pnt-1]**3 - beta_ * c[pnt-1])
        s[pnt]  = smax / (1 + (c[pnt] / hillaffinity)**hillcoef) if feedback else s[0]
        gg[pnt] = gg[pnt-1] + dt * (s[pnt-1] - p[pnt-1] * gg[pnt-1])
    return r, p, c, s, gg, cgmp2cur * gg**3

r_f, p_f, c_f, s_f, g_f, cur_f = phototransduction()

fig, axes = plt.subplots(5, 1, figsize=(7.5, 11), sharex=True)
for ax, dat, lab, col in [
        (axes[0], cur_f, "current (pA)",   "#2f6fb5"),
        (axes[1], p_f,   "pde activity",   "#7a5195"),
        (axes[2], s_f,   "synthesis rate", "#bc5090"),
        (axes[3], g_f,   "[cGMP]",         "#ef8354"),
        (axes[4], c_f,   "[calcium]",      "#3c896d")]:
    ax.plot(tmeP, dat, color=col)
    ax.set_ylabel(lab)
axes[0].set_title(rf"phototransduction with calcium feedback ($\beta$ = {beta_ca})")
axes[-1].set_xlabel("time (sec)")
fig.tight_layout()

print(f"beta = {beta_ca}  ->  calcium time constant 1/beta = {1/beta_ca:.1f} s, "
      f"but the simulation is only {NumPtsP*TimeStepP:.1f} s long")
print(f"calcium moves from {c_f[0]:.4f} to {c_f.min():.4f}  "
      f"({100*(1-c_f.min()/c_f[0]):.2f}% change)")
print(f"synthesis rate moves from {s_f[0]:.4f} to {s_f.max():.4f}  "
      f"({100*(s_f.max()/s_f[0]-1):.2f}% change)")

### The original parameter value does not actually show feedback

Read the printed numbers. With $\beta = 0.1\ \text{s}^{-1}$ the calcium time constant is
**10 seconds**, while the entire simulation is **1 second** long. Calcium barely moves,
the synthesis rate barely moves, and the "feedback" model is, over the window we plot,
indistinguishable from the no-feedback model of Part V. The plot looks fine, which is
exactly what makes it a trap: nothing warns you that the mechanism you just implemented is
switched off by its own time constant.

Real photoreceptor calcium turns over in tens to a couple of hundred milliseconds, i.e.
$\beta$ of order $5$–$50\ \text{s}^{-1}$. The homework question in the original file asks
"the model will generate damped oscillations for some values of $\beta$ — why?", but at
$\beta = 0.1$ you cannot see any feedback at all, let alone oscillations. So let's sweep
$\beta$, and use a **bright flash** ($r(0) = 20$) over a longer window so the feedback has
something to work against.

In [ ]:
NLONG = 3000
tmeL  = (np.arange(NLONG) + 1) * TimeStepP

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

summary = []
for beta_, col in zip([0.1, 1.0, 5.0, 20.0, 100.0],
                      plt.cm.viridis(np.linspace(0.1, 0.88, 5))):
    _, _, cc, ss, _, cu = phototransduction(beta_=beta_, r0=20.0, n=NLONG)
    axes[0].plot(tmeL, cu, color=col, label=rf"$\beta$ = {beta_:g}")
    axes[1].plot(tmeL, cc, color=col)
    imin = cu.argmin()
    summary.append((beta_, cu.min(), cu[imin:].max(), tmeL[imin]))

_, _, _, _, _, cu_nofb = phototransduction(r0=20.0, n=NLONG, feedback=False)
axes[0].plot(tmeL, cu_nofb, "k--", lw=1.2, label="no feedback")
axes[0].axhline(cgmp2cur * gdark**3, color="0.6", lw=0.8)
axes[0].set(ylabel="current (pA)",
            title="bright flash ($r_0$ = 20): calcium feedback vs. $\\beta$")
axes[0].legend(frameon=False, fontsize=9, ncol=2)
axes[1].axhline(cdark, color="0.6", lw=0.8)
axes[1].set(xlabel="time (sec)", ylabel="[calcium]")
for ax in axes:
    ax.set_xlim(0, tmeL[-1])
fig.tight_layout()

print(f"dark current = {cgmp2cur*gdark**3:.3f} pA\n")
print(f"{'beta':>7} {'min current':>12} {'time of min':>12} {'peak after min':>16}")
for b_, mn, mx, tmin in summary:
    print(f"{b_:>7g} {mn:>12.3f} {tmin*1000:>10.0f} ms {mx:>16.3f}")
print(f"{'none':>7} {cu_nofb.min():>12.3f} {tmeL[cu_nofb.argmin()]*1000:>10.0f} ms "
      f"{cu_nofb[cu_nofb.argmin():].max():>16.3f}")

Now the mechanism is visible. Compare each colored trace with the black dashed
no-feedback trace and with the grey line marking the dark current:

- **Feedback reduces the response amplitude.** The minimum current rises monotonically
  with $\beta$ — from 17.3 pA with no feedback and at $\beta=0.1$, up through 20.5 pA at
  $\beta=5$, to 22.7 pA at $\beta=100$ (see the printed table). Faster calcium removal
  means the synthesis rate climbs sooner during the response, opposing the cGMP drop while
  it is still happening.
- **Feedback speeds recovery, and makes the response briefer.** The time of the current
  minimum shortens from 557 ms with no feedback to about 205 ms at $\beta = 20$, and the
  colored traces climb back to the dark line (grey) while the no-feedback trace is still
  far below it at 3 s.
- **At intermediate $\beta$ the recovery overshoots and rings.** This is the damped
  oscillation the original homework asks about, and it is clearest at $\beta = 1$: the
  current shoots past the dark level to 32 pA at about 1.4 s, then dips back below it, and
  the calcium trace shows the same ringing one quarter-cycle out of phase. At $\beta = 5$
  the overshoot is smaller (28.1 pA) and faster. At $\beta = 20$ and $100$ there is no
  overshoot at all — the current settles onto the dark line and stays there.

Why does the *middle* of the $\beta$ range ring? Because the loop is negative feedback with
a **delay**: calcium reports the current from roughly $1/\beta$ seconds ago, so the
correction arrives late, overshoots, is corrected in the other direction, and so on. When
$\beta$ is large the calcium signal is essentially instantaneous, the delay vanishes, and
the loop settles cleanly. When $\beta$ is small the loop still overshoots — the $\beta=0.1$
trace does eventually rise above the dark level — but it does so on a 10-second timescale,
so slowly that over the duration of a real light response the feedback is effectively
absent. That is exactly the trap in the previous figure.

Delayed negative feedback producing damped oscillation is not special to photoreceptors —
it is the same phenomenon as a thermostat that overshoots because the thermometer is on
the far wall, and the same as the ringing you get in a poorly tuned control loop.

> ### Homework question 5
> **(a)** Explain the relation between the steady-state conditions and the constants $q$
> and $s_{\max}$. Why are they not free parameters?
>
> **(b)** Play with the various parameters (`hillcoef`, `hillaffinity`, `sigma`, `phi`,
> `gdark`) and see how they alter the calculated light response. Explain why things change
> the way they do.
>
> **(c)** The model will generate damped oscillations for some values of $\beta$. Why? Use
> the $\beta$ sweep above, and say what sets the *period* of the oscillation.
>
> **(d)** What does the Hill coefficient $h$ control? Predict, then check, what happens as
> $h \to 0$ and as $h$ becomes large.

---
## Part VII. Two-state systems: Hodgkin–Huxley gating particles

The Hodgkin–Huxley model relies on a series of **gating particles**, each of which can be
active or inactive. Let's consider one of them: the $m$ gate controlling sodium channel
opening.

Write $m$ for the probability that the particle is in the active state; then $1-m$ is the
probability it is inactive. The rate of change of $m$ is the likelihood that an inactive
particle becomes active, $\alpha (1-m)$, minus the likelihood that an active particle
inactivates, $\beta m$:

$$\frac{dm}{dt} = \alpha\,(1-m) \;-\; \beta\,m.$$

Here $\alpha$ is the **inactive $\to$ active** rate and $\beta$ is the **active $\to$
inactive** rate. (The comments in the MATLAB original label these the other way round;
compare them with the equation itself and you will see the labels are swapped.)

Rearranging shows that this is again a single exponential:

$$\frac{dm}{dt} = \frac{m_\infty - m}{\tau_m}, \qquad
m_\infty = \frac{\alpha}{\alpha + \beta}, \qquad
\tau_m = \frac{1}{\alpha + \beta}.$$

So a two-state system always relaxes exponentially toward $m_\infty$ with time constant
$\tau_m$ — which is why so much of channel biophysics is described by pairs
$(m_\infty, \tau_m)$ rather than by $(\alpha, \beta)$.

In [ ]:
alpha_m = 200.0         # inactive -> active rate constant, 1/sec
beta_m  = 100.0         # active -> inactive rate constant, 1/sec
NumPtsM = 1000
TimeStepM = 1e-5        # 10 us steps -> 10 ms of simulated time

m = np.zeros(NumPtsM)
m[0] = 1.0              # start fully active, and watch it relax

for pnt in range(1, NumPtsM):
    m[pnt] = m[pnt-1] + TimeStepM * ((1 - m[pnt-1]) * alpha_m - m[pnt-1] * beta_m)

tmeM  = (np.arange(NumPtsM) + 1) * TimeStepM
minf  = alpha_m / (alpha_m + beta_m)
tau_m = 1.0 / (alpha_m + beta_m)
m_an  = minf + (m[0] - minf) * np.exp(-tmeM / tau_m)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(tmeM * 1000, m, color="#2f6fb5", label="Euler")
ax.plot(tmeM * 1000, m_an, "--", color="#cc5544", lw=1.2,
        label=r"analytical $m_\infty + (m_0-m_\infty)e^{-t/\tau_m}$")
ax.axhline(minf, color="0.6", lw=0.8)
ax.annotate(rf"$m_\infty$ = {minf:.3f}", xy=(7, minf), xytext=(7, minf + 0.06),
            color="0.35", fontsize=9)
ax.set(xlabel="time (ms)", ylabel="$m$", ylim=(0, 1.05),
       title="two-state gating particle, constant rates")
ax.legend(frameon=False)
fig.tight_layout()

print(f"m_inf = alpha/(alpha+beta) = {minf:.4f}")
print(f"m at the end of the 10 ms run = {m[-1]:.4f}  (3 time constants -> not quite there yet)")
print(f"tau_m = 1/(alpha+beta)     = {tau_m*1000:.3f} ms")
print(f"(alpha+beta)*TimeStep      = {(alpha_m+beta_m)*TimeStepM:.4f}  (tiny -> Euler is very accurate)")
print(f"max |Euler - analytical|   = {np.max(np.abs(m - m_an)):.2e}")

The numerical and analytical curves are indistinguishable here, and the printed maximum
error is about $10^{-3}$ — around 0.3% of the excursion, against roughly 18% in Part I.
The reason is entirely the step size: $(\alpha+\beta)\Delta t \approx 0.003$ here, versus
$\alpha \Delta t = 0.2$ there. This is what "small enough time step" looks like.

(The run stops after 10 ms, which is only three time constants, so $m$ has reached 0.683
rather than settling exactly onto $m_\infty = 0.667$. That is the physics, not the
integrator.)

### Voltage-dependent rates

The interesting part of Hodgkin–Huxley is that $\alpha$ and $\beta$ are **functions of
membrane voltage**. For the sodium activation gate the course's parameterization is

$$\alpha_m(V) = \frac{-100\,(V+30)}{\exp\!\bigl(-(V+30)/10\bigr) - 1},
\qquad
\beta_m(V) = 4000\, \exp\!\bigl(-(V+55)/18\bigr),$$

with $V$ in mV and both rates in s$^{-1}$. Note that $\alpha_m$ is $0/0$ at exactly
$V = -30$ mV — the numerator and denominator both vanish. In MATLAB you get a `NaN`; in
NumPy you get a `NaN` and a warning. The function has a perfectly good finite limit
there ($\alpha_m = 1000\ \text{s}^{-1}$, by L'Hôpital), so we handle that point
explicitly rather than letting a `NaN` propagate through the whole simulation.

> **A bug in the MATLAB original.** The MATLAB file computes
> `alpha(1:NumPts) = -100 * (v + 30) / (exp(-(v + 30)/10) - 1);`
> with a plain `/`, not `./`. With two row vectors, MATLAB's `/` is *matrix right
> division* — a least-squares solve that returns a single **scalar**, silently. So in the
> original, $\alpha$ ends up constant with voltage while $\beta$ (which uses a scalar
> multiply and is therefore element-wise) is correctly voltage-dependent. The figure it
> produces is not the figure its comments describe. We use element-wise arithmetic, so
> both rates vary with voltage as intended.

In [ ]:
def alpha_m_of_V(v):
    '''HH sodium activation rate, 1/sec. Handles the removable singularity at V = -30 mV.'''
    v = np.asarray(v, dtype=float)
    num = -100.0 * (v + 30.0)
    den = np.exp(-(v + 30.0) / 10.0) - 1.0
    # at v = -30 both vanish; the limit is 100 * 10 = 1000 /sec
    return np.where(np.abs(v + 30.0) < 1e-9, 1000.0, num / np.where(den == 0, np.nan, den))

def beta_m_of_V(v):
    '''HH sodium deactivation rate, 1/sec.'''
    return 4000.0 * np.exp(-(np.asarray(v, dtype=float) + 55.0) / 18.0)

PrePtsM = 200           # points before the voltage step
StmPtsM = 400           # points during the step

# voltage: step from -60 to -40 mV
v = np.full(NumPtsM, -60.0)
v[PrePtsM:PrePtsM + StmPtsM] = -40.0

alpha_v = alpha_m_of_V(v)       # element-wise -- see the note above about MATLAB's `/`
beta_v  = beta_m_of_V(v)

mv = np.zeros(NumPtsM)
mv[0] = 0.0                     # the original's initial condition (not the resting value!)
for pnt in range(1, NumPtsM):
    mv[pnt] = mv[pnt-1] + TimeStepM * ((1 - mv[pnt-1]) * alpha_v[pnt-1]
                                       - mv[pnt-1] * beta_v[pnt-1])

# the same run started from rest at -60 mV, so the step response stands alone
mv_rest = np.zeros(NumPtsM)
mv_rest[0] = alpha_v[0] / (alpha_v[0] + beta_v[0])
for pnt in range(1, NumPtsM):
    mv_rest[pnt] = mv_rest[pnt-1] + TimeStepM * ((1 - mv_rest[pnt-1]) * alpha_v[pnt-1]
                                                 - mv_rest[pnt-1] * beta_v[pnt-1])

fig, axes = plt.subplots(3, 1, figsize=(7.5, 8), sharex=True)
axes[0].plot(tmeM * 1000, v, color="#333333")
axes[0].set(ylabel="V (mV)", ylim=(-65, -35),
            title="sodium activation gate, voltage-dependent rates")
axes[1].plot(tmeM * 1000, alpha_v, color="#2f6fb5", label=r"$\alpha_m(V)$")
axes[1].plot(tmeM * 1000, beta_v,  color="#cc5544", label=r"$\beta_m(V)$")
axes[1].set(ylabel="rate (1/s)")
axes[1].legend(frameon=False)
axes[2].plot(tmeM * 1000, mv, color="#2f6fb5", label="$m(0)=0$ (original)")
axes[2].plot(tmeM * 1000, mv_rest, color="#3c896d", label="$m(0)=m_\\infty(-60)$")
axes[2].plot(tmeM * 1000, alpha_v / (alpha_v + beta_v), ":", color="0.5",
             label=r"$m_\infty(V)$")
axes[2].set(xlabel="time (ms)", ylabel="$m$")
axes[2].legend(frameon=False, fontsize=9)
fig.tight_layout()

for vv in (-60.0, -40.0):
    a_, b_ = float(alpha_m_of_V(vv)), float(beta_m_of_V(vv))
    print(f"V = {vv:>5.0f} mV :  alpha = {a_:8.1f} /s   beta = {b_:8.1f} /s   "
          f"m_inf = {a_/(a_+b_):.4f}   tau = {1000/(a_+b_):.3f} ms")
print(f"\npeak m during the step (from rest) = {mv_rest[PrePtsM:PrePtsM+StmPtsM].max():.4f}")
print(f"largest rate * TimeStep = {beta_v.max()*TimeStepM:.4f}  (well below 2 -> stable)")

The two initial conditions matter. Started at $m(0)=0$ (the original's choice), the trace
spends the first couple of milliseconds relaxing up to the resting $m_\infty(-60)$; that
early rise is a simulation artifact, not a response to the step. Started from rest (green)
the step response stands on its own.

The dotted grey line is $m_\infty(V)$ — where $m$ would go if it had infinite time at each
voltage. $m$ tracks it closely because $\tau_m$ is a few hundred microseconds, far shorter
than the 4 ms step. This is exactly why the sodium activation gate is treated as fast:
it is essentially at its steady state at all times on the timescale of an action potential,
while the slower $h$ and $n$ gates are not.

> ### Extensions (not problems to be turned in)
> **(a)** Add an inactivation gating particle $h$, with
> $\alpha_h = 70\exp(-(V+55)/20)$ and $\beta_h = 1000/[\exp(-(V+25)/10)+1]$ (per second),
> and plot $m^3 h$ — the sodium conductance's gating term.
>
> **(b)** Build this up to the full Hodgkin–Huxley sodium current model, then add the
> potassium $n$ gate and the leak, and produce an action potential. You now have every
> tool you need: it is four coupled equations of exactly the form you have been solving.

---
## Part VIII. Solving differential equations with Fourier transforms

Fourier transforms provide a completely different and often very elegant route to solving
differential equations. The key trick is that **derivatives turn into multiplications**
when you take the Fourier transform. With the convention used by `numpy.fft` (and by
MATLAB's `fft`),

$$X(f) = \int x(t)\, e^{-2\pi i f t}\, dt \qquad\Longrightarrow\qquad
\frac{dx}{dt} \;\longleftrightarrow\; 2\pi i f\, X(f).$$

(Make sure you can get this from the definition of the Fourier transform by integrating by
parts. The MATLAB comments write the factor as $-2i\omega$, which mixes up both the sign
and the $\omega$-vs-$f$ convention; the code itself uses $+2\pi i f$, which is the correct
one for `fft`. This is a standard trap — always derive the factor from the transform
convention your library actually uses.)

Apply this to the very first equation we considered,

$$\frac{dx}{dt} = y - \alpha x.$$

Transforming both sides:

$$2\pi i f\, X(f) = Y(f) - \alpha X(f)
\qquad\Longrightarrow\qquad
\boxed{\,X(f) = \frac{Y(f)}{\alpha + 2\pi i f}\,}$$

The differential equation has become **division**. Invert the transform and you have the
answer. Notice what we got for free: an analytical statement that the system is a low-pass
filter with corner frequency $\alpha/2\pi$, valid for *any* input. That is the real payoff
— you can see how the solution depends on $\alpha$ without running a single simulation.

Let's check it against the Euler solution.

In [ ]:
alpha_F   = 20.0
TimeStepF = 0.001
PrePtsF, StmPtsF, NumPtsF = 200, 400, 1000

yF = np.zeros(NumPtsF)
yF[PrePtsF:PrePtsF + StmPtsF] = 1.0
tmeF = (np.arange(NumPtsF) + 1 - PrePtsF) * TimeStepF

# --- (1) time domain: the Euler loop, as before ---
xF = np.zeros(NumPtsF)
for pnt in range(1, NumPtsF):
    xF[pnt] = xF[pnt-1] + (yF[pnt-1] - alpha_F * xF[pnt-1]) * TimeStepF

# --- (2) frequency domain ---
Y = np.fft.fft(yF)

# Frequency axis. numpy.fft.fftfreq builds exactly the ordering MATLAB's fft uses:
# 0, +df, ... up to just under Nyquist, then the negative frequencies from most
# negative back toward zero. (The MATLAB original constructs it by hand; fftfreq
# is one call and removes the off-by-one risk.)
Freq = np.fft.fftfreq(NumPtsF, d=TimeStepF)

X  = Y / (alpha_F + 2j * np.pi * Freq)
xF_fft = np.fft.ifft(X)

fig, axes = plt.subplots(3, 1, figsize=(7.5, 8.5), sharex=True)
axes[0].plot(tmeF, yF, color="#333333")
axes[0].set(ylim=(-0.1, 1.1), ylabel="input $y$",
            title="the same equation, two ways")
axes[1].plot(tmeF, xF, color="#2f6fb5")
axes[1].set(ylabel="output $x$ (Euler)")
axes[2].plot(tmeF, xF_fft.real, color="#bc5090")
axes[2].set(xlabel="time (sec)", ylabel="output $x$ (Fourier)")
# panels 2 and 3 are meant to be compared, so give them identical limits
for ax in axes[1:]:
    ax.set_ylim(-0.005, 0.06)
fig.tight_layout()

print(f"largest imaginary part of ifft result = {np.max(np.abs(xF_fft.imag)):.3e} "
      "(round-off; the answer is real, as it must be)")
print(f"Euler peak   = {xF.max():.6f}")
print(f"Fourier peak = {xF_fft.real.max():.6f}")
print(f"max |Euler - Fourier| = {np.max(np.abs(xF - xF_fft.real)):.6f} "
      f"({100*np.max(np.abs(xF - xF_fft.real))/xF.max():.2f}% of peak)")

The two approaches give nearly the same answer — the printed difference is about 1% of the
peak, and it is dominated by the Euler solution's first-order lag, not by anything wrong
with the Fourier method. (Note the Fourier peak, 0.05004, sits slightly *above* the exact
steady state $1/\alpha = 0.05$, while the Euler peak, 0.04999, sits slightly below.)

Two caveats worth internalizing, because they catch people:

1. **The `ifft` result is complex.** Mathematically it must be real, since the input was
   real and the transfer function is Hermitian; what you actually get is real plus
   round-off at the $10^{-19}$ level. MATLAB silently plots the real part and issues a
   warning; in Python you take `.real` yourself. Never take `abs()` — that would flip the
   sign of any negative excursion.

2. **The Fourier solution is periodic.** The FFT treats your signal as one period of an
   infinitely repeating waveform, so the response wraps around from the end of the trace
   back to the beginning. Here $\tau = 1/\alpha = 50$ ms and the trace is 1 s long, so
   everything has decayed long before the wrap and the two solutions agree. Shrink
   `NumPtsF`, or make $\alpha$ small, and you will see the wraparound contaminate the
   start of the trace. The Fourier method also gives you no control over the initial
   condition: it returns the *steady-state periodic* solution, not the solution from a
   particular $x(0)$.

The power of this approach is that many differential equations you cannot guess the answer
to in the time domain can be solved in the frequency domain. Having an analytical solution
lets you identify how the answer depends on a particular parameter, instead of extracting
that by running many numerical solutions. The catch: it works only for **linear**
equations with constant coefficients. The feedback model of Part III, and the
phototransduction model of Part VI, cannot be touched this way — which is why we went to
all that trouble with `for` loops.

---
## Part IX. How wrong is Euler, and when does it explode?

This section is a Python-side addition, but it answers a question the original tutorial
asks three separate times: over what range of time steps is the numerical solution
accurate?

Take the simplest possible case, where we know the answer exactly:

$$\frac{dr}{dt} = -\sigma r, \qquad r(0) = 1, \qquad r(t) = e^{-\sigma t}.$$

One Euler step gives $r(n) = r(n-1)(1 - \sigma \Delta t)$, so after $N$ steps
$r = (1-\sigma\Delta t)^N$, whereas the true answer is $e^{-\sigma N \Delta t}$. Expanding
both shows the discrepancy is $O(\Delta t)$ per unit time: **forward Euler is a
first-order method**. Halve the step, halve the error.

The factor $(1-\sigma\Delta t)$ also tells you exactly when Euler blows up. The numerical
solution stays bounded only if $|1 - \sigma\Delta t| \le 1$, i.e.

$$\sigma \Delta t \le 2.$$

Between $\sigma\Delta t = 1$ and $2$ the solution alternates sign each step while shrinking
(wrong, but bounded); beyond $2$ it alternates sign and **grows without limit**. The
underlying differential equation has no such behavior at all — the instability is entirely
manufactured by the integration scheme.

We check both claims, and compare against `scipy.integrate.solve_ivp`.

In [ ]:
sigma_t = 5.0
T_end   = 1.0

def euler_decay(dt, sigma_=sigma_t, T=T_end):
    n = int(round(T / dt)) + 1
    t = np.arange(n) * dt
    rr = np.zeros(n); rr[0] = 1.0
    for i in range(1, n):
        rr[i] = rr[i-1] + dt * (-sigma_ * rr[i-1])
    return t, rr

dts   = np.array([0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 5e-4, 2e-4, 1e-4])
err_e = np.array([np.max(np.abs(r_ - np.exp(-sigma_t * t_)))
                  for t_, r_ in (euler_decay(d) for d in dts)])

# scipy's adaptive Runge-Kutta on exactly the same problem
sol = solve_ivp(lambda t, yv: -sigma_t * yv, (0, T_end), [1.0],
                method="RK45", rtol=1e-10, atol=1e-12, dense_output=True)
t_fine  = np.linspace(0, T_end, 2001)
err_rk  = np.max(np.abs(sol.sol(t_fine)[0] - np.exp(-sigma_t * t_fine)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

axes[0].loglog(dts, err_e, "o-", color="#2f6fb5", label="forward Euler")
axes[0].loglog(dts, err_e[-1] * (dts / dts[-1]), "--", color="0.5",
               label=r"slope 1 ($\propto \Delta t$)")
axes[0].axhline(err_rk, color="#cc5544", lw=1.4,
                label=f"solve_ivp RK45 ({err_rk:.1e})")
axes[0].set(xlabel=r"time step $\Delta t$ (s)", ylabel="max |numerical - exact|",
            title=r"first-order convergence, $dr/dt=-\sigma r$")
axes[0].legend(frameon=False, fontsize=9)

for dt_, col in zip([0.05, 0.2, 0.35, 0.45],
                    ["#3c896d", "#2f6fb5", "#ef8354", "#cc2222"]):
    t_, r_ = euler_decay(dt_, T=2.0)
    axes[1].plot(t_, r_, "o-", ms=3, color=col,
                 label=rf"$\Delta t$={dt_:g}, $\sigma\Delta t$={sigma_t*dt_:.2f}")
tt = np.linspace(0, 2, 400)
axes[1].plot(tt, np.exp(-sigma_t * tt), "k--", lw=1.2, label="exact")
axes[1].axhline(0, color="0.7", lw=0.8)
axes[1].set(xlabel="time (s)", ylabel="$r$", ylim=(-2.5, 2.5),
            title="stability: Euler explodes past $\\sigma\\Delta t = 2$")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()

print("forward Euler, max absolute error vs. time step:")
for d, e in zip(dts, err_e):
    print(f"   dt = {d:8.5f}  (sigma*dt = {sigma_t*d:6.3f})   error = {e:.3e}")
print(f"\nhalving dt from {dts[4]:g} to {dts[5]:g} changes the error by a factor "
      f"{err_e[4]/err_e[5]:.2f}  (first order => ~2)")
print(f"solve_ivp RK45 (rtol=1e-10) error = {err_rk:.2e}")

The left panel is the convergence statement: the error tracks the dashed slope-1 reference
almost perfectly over four decades of step size, and halving $\Delta t$ halves the error —
the printed ratio is close to 2, as it must be for a first-order method. `solve_ivp`'s
adaptive fifth-order Runge–Kutta sits many orders of magnitude below, at essentially
machine precision, for a comparable amount of work.

The right panel is the stability statement. At $\sigma\Delta t = 0.25$ the solution decays
sensibly. At exactly $\sigma\Delta t = 1.00$ the update factor $(1-\sigma\Delta t)$ is
zero, so the solution jumps to zero in a single step and stays there — stable, but a
caricature of an exponential. At $1.75$ it alternates sign every step while still
shrinking: bounded, and completely wrong. At $2.25$ it alternates sign and **grows** — an
exponentially *decaying* process, integrated into an exponentially *growing* answer.

Two lessons for practice:

- **Choose $\Delta t$ from the fastest rate constant in the model, not from the timescale
  you care about.** In the HH section the fastest rate was $\beta_m \approx 5300\ \text{s}^{-1}$,
  requiring $\Delta t \ll 0.4$ ms; the tutorial uses 10 µs. In the phototransduction model
  with large $\beta$, calcium is the fast variable and sets the limit.
- **A stable-looking answer is not a correct answer.** All the runs earlier in this
  tutorial were comfortably stable, but the Part I solution still carried a few percent
  error. Stability is a much weaker condition than accuracy.

Now let's confirm that the Euler results earlier in the tutorial were trustworthy, by
re-solving the full phototransduction model with `solve_ivp` and overlaying the two. Note
how the ODE system is written for `solve_ivp`: a function returning the vector of
derivatives, in exactly the form you have been writing inside your loops.

In [ ]:
beta_chk = 20.0     # a beta where the feedback is actually engaged
cur2ca_c = beta_chk * cdark / (cgmp2cur * gdark**3)
smax_c   = eta/phi * gdark * (1 + (cdark / hillaffinity)**hillcoef)

def photo_rhs(t, Y):
    '''State Y = [r, p, c, g]; returns dY/dt. Same equations as the Euler loop.'''
    r_, p_, c_, g_v = Y
    s_v = smax_c / (1 + (c_ / hillaffinity)**hillcoef)
    return [-sigma * r_,
            r_ + eta - phi * p_,
            cur2ca_c * cgmp2cur * g_v**3 - beta_chk * c_,
            s_v - p_ * g_v]

Y0 = [20.0, eta/phi, cdark, gdark]
sol_p = solve_ivp(photo_rhs, (0, 3.0), Y0, method="LSODA",
                  rtol=1e-9, atol=1e-11, dense_output=True)

fig, ax = plt.subplots(figsize=(8, 4.6))
tref = np.linspace(0, 3.0, 3000)
ax.plot(tref, cgmp2cur * sol_p.sol(tref)[3]**3, "k-", lw=2.4, alpha=0.35,
        label="solve_ivp (LSODA, reference)")
for dt_, col in zip([0.005, 0.002, 0.001], ["#ef8354", "#7a5195", "#2f6fb5"]):
    n_ = int(round(3.0 / dt_))
    _, _, _, _, _, cu_ = phototransduction(beta_=beta_chk, r0=20.0, n=n_, dt=dt_)
    t_ = (np.arange(n_) + 1) * dt_
    ax.plot(t_, cu_, color=col, lw=1.2, label=rf"Euler, $\Delta t$ = {dt_*1000:g} ms")
ax.set(xlabel="time (sec)", ylabel="current (pA)", xlim=(0, 3.0),
       title=rf"Euler vs. solve_ivp, full model ($\beta$ = {beta_chk:g}, bright flash)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

ref = cgmp2cur * sol_p.sol(tref)[3]**3
print("max |Euler - solve_ivp| on the current trace (interpolated to common times):")
for dt_ in [0.005, 0.002, 0.001, 0.0005]:
    n_ = int(round(3.0 / dt_))
    _, _, _, _, _, cu_ = phototransduction(beta_=beta_chk, r0=20.0, n=n_, dt=dt_)
    t_ = (np.arange(n_) + 1) * dt_
    e_ = np.max(np.abs(np.interp(tref, t_, cu_) - ref))
    print(f"   dt = {dt_*1000:5.2f} ms   max error = {e_:.4f} pA "
          f"({100*e_/ref.max():.2f}% of the dark current)")

At the tutorial's default $\Delta t = 1$ ms the Euler solution is visually on top of the
reference and the printed error is a fraction of a percent of the dark current. So yes:
the numbers you were looking at in Parts IV–VI were trustworthy. But the errors printed
above scale the way Part IX predicts, and the coarsest step is already visibly off during
the fast falling phase, where the derivative is largest.

The general rule: use `solve_ivp` for real work — it is adaptive, higher order, and has
stiff solvers (`LSODA`, `Radau`, `BDF`) for systems with widely separated rate constants
like this one. Write the Euler loop when you want to *understand* what the solver is
doing, or when you need the state at every fixed time bin (which is often the case when
you are comparing a model to sampled data).

---
## Part X. Coupled systems: an oculomotor plant model

Everything so far has been a scalar equation or a chain of them. Real mechanical systems
give you **coupled second-order** equations, and the standard trick — the one `solve_ivp`
requires — is to rewrite an $n$-th order equation as $n$ first-order equations by making
the derivatives into state variables.

The course directory contains two helper functions, `dydt.m` and `dydt10.m`, that do
exactly this for a **linear oculomotor plant**: a model of the eye and its muscles as a
mass, a viscous element, and springs, driven by the muscle's active-state tension. (Note
that `DiffEQTutorial.m` never actually calls them — they are set up for a separate
eye-movement exercise — but they are the natural next step from this tutorial, and they
are what a derivative function looks like once the state has more than one component.)

The state vector is
$$\mathbf{y} = \bigl[\theta,\; \dot\theta,\; F_m,\; F_p \bigr],$$
eye position (deg), eye velocity (deg/s), the force in the muscle's series element, and
the force in the passive (parallel) element. With $m$ the inertia, $R_m$ and $K_e$ the
muscle's viscous and elastic elements, $F_0(t)$ the active-state tension and $F_a(t)$ an
externally applied load, the model of `dydt.m` is

$$\begin{aligned}
\dot\theta &= \dot\theta, \\[2pt]
\ddot\theta &= \frac{F_m + F_a - F_p}{m}, \\[2pt]
\dot F_m &= \frac{K_e}{R_m}\left( F_0 - R_m \dot\theta - F_m \right), \\[2pt]
\dot F_p &= \frac{R_1 R_2 \dfrac{F_m + F_a - F_p}{m}
   + (R_1 K_2 + R_2 K_1)\dot\theta + K_1 K_2\,\theta - (K_1+K_2) F_p}{R_1 + R_2}.
\end{aligned}$$

The last line is the "two-Voigt-element" arrangement of the passive tissue — two
spring-and-dashpot pairs in series. `dydt.m` also offers simpler passive arrangements
(a single Voigt element, a pure dashpot, a pure spring) selected by a `mode` argument, and
several loading conditions selected by an `option` argument; the isometric condition
clamps the eye by setting $F_a = -K_i \theta$. `dydt10.m` is the same function hard-wired
to the two-Voigt case. We port both in one Python function with the same switches.

Note the two MATLAB-to-Python details in the port: `interp1(commandTime, F0, t)` becomes
`np.interp`, and `y(1)`…`y(4)` become `y[0]`…`y[3]`.

In [ ]:
# --- Plant parameters, straight from dydt.m (units in the comments) ---
mi = np.array([0.677e-4,    # g*sec^2/deg, normal
               2.16e-4,     # g*sec^2/deg, isometric saccade
               28.9e-4])    # g*sec^2/deg, with sled load
Rm = 0.072      # g*sec/deg   muscle series viscosity
Ke = 3.6        # g/deg       muscle series elasticity
K1, R1 = 2.06, 0.025        # passive Voigt element 1
K2, R2 = 6.36, 1.81         # passive Voigt element 2
Ki = 15.0       # g/deg       stiffness of the isometric clamp

def dydt(t, y, commandTime, F0_cmd, Fa_cmd, option="normal", mode="two_voigt"):
    '''Port of dydt.m / dydt10.m.  y = [theta, theta_dot, Fm, Fp]; returns dy/dt.

    MATLAB's interp1 -> np.interp.  MATLAB's y(1)..y(4) -> y[0]..y[3].
    `mode` selects the passive-element arrangement; `option` selects the load.
    '''
    F0 = np.interp(t, commandTime, F0_cmd)      # active-state tension, g
    Fa = np.interp(t, commandTime, Fa_cmd)      # applied load, g

    m = mi[0]
    if option == "isometric":
        m  = mi[1]
        Fa = -Ki * y[0]                         # the clamp reacts against eye position
    elif option == "high inertial":
        m = mi[2]
    # 'normal' and 'isotonic' both use mi[0]

    accel = (y[2] + Fa - y[3]) / m              # theta'' , shared by every mode
    dFm   = (F0 - Rm * y[1] - y[2]) * Ke / Rm

    if mode == "one_voigt":
        K, R = 2.06, 0.025
        dFp = R / m * (y[2] - y[3] + Fa) + K * y[1]
    elif mode == "dashpot":
        R = 0.3
        dFp = R / m * (y[2] - y[3] + Fa)
    elif mode == "spring":
        K = 2.06
        dFp = K * y[1]
    elif mode == "muscle_voigt":
        # this mode also replaces the theta'' equation (see dydt.m)
        Kv, Rv = 1.0, 0.05
        accel = (-2 * Kv * y[1] - Kv / Rv * y[2] + (Kv / Rv - 1) * F0) / Rv
        dFp = (R1*R2*(y[2] + Fa - y[3])/m + (R1*K2 + R2*K1)*y[1]
               + K1*K2*y[0] - (K1 + K2)*y[3]) / (R1 + R2)
    else:  # 'two_voigt', the arrangement hard-wired into dydt10.m
        dFp = (R1*R2*(y[2] + Fa - y[3])/m + (R1*K2 + R2*K1)*y[1]
               + K1*K2*y[0] - (K1 + K2)*y[3]) / (R1 + R2)

    return [y[1], accel, dFm, dFp]

# --- A pulse-step command: the classic saccadic drive signal ---
commandTime = np.linspace(0, 0.4, 4001)
F0_cmd = np.full_like(commandTime, 0.0)
pulse  = (commandTime >= 0.05) & (commandTime < 0.06)   # 10 ms pulse
step   = commandTime >= 0.06
F0_cmd[pulse] = 60.0     # g, large transient drive
F0_cmd[step]  = 20.0     # g, smaller maintained drive holds the new position
Fa_cmd = np.zeros_like(commandTime)

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].plot(commandTime * 1000, F0_cmd, color="#333333")
axes[0].set(ylabel="$F_0$ (g)", title="oculomotor plant: pulse-step drive")

for mode, col in [("two_voigt", "#2f6fb5"), ("one_voigt", "#7a5195"),
                  ("dashpot", "#ef8354"), ("spring", "#3c896d")]:
    s = solve_ivp(dydt, (0, 0.4), [0.0, 0.0, 0.0, 0.0],
                  args=(commandTime, F0_cmd, Fa_cmd, "normal", mode),
                  method="LSODA", rtol=1e-8, atol=1e-10, dense_output=True)
    tt = np.linspace(0, 0.4, 2000)
    Y  = s.sol(tt)
    axes[1].plot(tt * 1000, Y[0], color=col, label=mode)
    axes[2].plot(tt * 1000, Y[1], color=col)

axes[1].set(ylabel=r"eye position $\theta$ (deg)")
axes[1].legend(frameon=False, fontsize=9, title="passive element")
axes[2].set(xlabel="time (ms)", ylabel=r"eye velocity $\dot\theta$ (deg/s)")
fig.tight_layout()

s2 = solve_ivp(dydt, (0, 0.4), [0.0, 0.0, 0.0, 0.0],
               args=(commandTime, F0_cmd, Fa_cmd, "normal", "two_voigt"),
               method="LSODA", rtol=1e-8, atol=1e-10, dense_output=True)
tt = np.linspace(0, 0.4, 2000); Y2 = s2.sol(tt)
print(f"two_voigt: final eye position   = {Y2[0][-1]:.2f} deg")
print(f"two_voigt: peak velocity        = {Y2[1].max():.0f} deg/s "
      f"at t = {tt[Y2[1].argmax()]*1000:.0f} ms")
print(f"solve_ivp took {s2.nfev} function evaluations for 400 ms of simulated time")

The pulse-step drive is the classic saccadic command: a large brief **pulse** of force to
overcome the eye's viscosity and move it fast, followed by a smaller maintained **step** to
hold it against the elastic restoring force. Get the pulse-step ratio wrong and the eye
either drifts back after the saccade or overshoots — a clinically recognizable pattern.

The passive-element arrangement changes the response qualitatively, and the differences
are a compact lesson in mechanics:

- **`spring`** (green): the passive tissue is a pure elastic element, with nothing to
  dissipate energy. The eye and its inertia form an undamped mass–spring oscillator, and
  the saccade sets it **ringing** — the velocity trace oscillates for hundreds of
  milliseconds. Real eyes plainly do not do this.
- **`dashpot`** (orange): a pure viscous element, with no restoring force. Nothing holds
  the eye anywhere, so it **drifts steadily** and never settles — position is still
  climbing linearly at 400 ms.
- **`one_voigt` / `two_voigt`** (purple, blue): a spring and a dashpot together, which is
  what a real tissue behaves like. These settle quickly with no ringing, and the two-Voigt
  version shows the slow post-saccadic **creep** — the eye keeps easing toward its final
  position for hundreds of milliseconds after the fast phase is over — that is a
  well-documented feature of real eye movements. This is why the model needs two Voigt
  elements with very different time constants ($R_1/K_1 \approx 12$ ms versus
  $R_2/K_2 \approx 285$ ms): one for the fast movement, one for the creep.

Peak velocity for the two-Voigt plant is about 440 deg/s for an 11.6 deg movement, in the
right range for a real saccade of that size.

Notice that we never wrote an integration loop here at all. Once you can write the
right-hand side as a function returning $d\mathbf{y}/dt$ — which is exactly what the body
of every `for` loop in this tutorial was doing — `solve_ivp` handles step-size selection,
error control and stiffness for you. The whole point of Parts I–IX was to make that
function, and its failure modes, not a mystery.

> ### Homework question 6
> **(a)** Run the plant with `option="isometric"`, which clamps the eye by setting
> $F_a = -K_i \theta$ and increases the effective inertia. What happens to the force in
> the series element, and why is this the condition experimenters use to measure muscle
> force directly?
>
> **(b)** Vary the pulse-step ratio (the relative heights of the 60 g pulse and the 20 g
> step). Find the ratio that produces neither post-saccadic drift nor overshoot. What does
> the mismatch look like in each direction?
>
> **(c)** Rewrite the `two_voigt` system as an explicit Euler loop and find the largest
> $\Delta t$ that stays stable. Compare that to the fastest rate constant in the model
> ($K_e/R_m$ is a good candidate). Does the prediction from Part IX hold?

---
## Summary

1. **Turn the derivative into a difference.** Replacing $dx/dt$ with
   $[x(n)-x(n-1)]/\Delta t$ and solving for $x(n)$ gives an update rule you can iterate.
   That is forward Euler, and it is the whole idea. Every model in this tutorial —
   two-state gates, second-messenger cascades, coupled mechanical systems — is the same
   three lines of code with a different right-hand side.

2. **A differential equation has no unique solution without an initial condition.**
   Choosing it badly produces a start-up transient that looks exactly like a real
   response. When your model should start at rest, solve for the steady state and start
   it there.

3. **Linear equations are forgiving; feedback is not.** Adding a constant to a linear
   equation shifts the answer by a constant (Part II). Adding feedback changes amplitude
   and kinetics differently, breaks superposition, and puts analytical solutions out of
   reach (Part III). That is precisely when numerical methods stop being a convenience
   and become the only option.

4. **Delayed negative feedback rings.** The calcium loop in phototransduction reduces the
   response amplitude and speeds recovery, and at intermediate feedback delays it produces
   damped oscillations — because the correction always arrives a little too late (Part VI).

5. **Fourier transforms turn derivatives into multiplications**, converting a linear
   constant-coefficient differential equation into a division (Part VIII). The payoff is
   an analytical answer, valid for any input, that shows you how the solution depends on
   each parameter. The price is that it works only for linear systems, returns the
   periodic steady-state solution, and gives you no control over the initial condition.

6. **Euler is first order and conditionally stable.** Error $\propto \Delta t$; the
   solution blows up once (rate) $\times \Delta t$ exceeds 2 (Part IX). Pick $\Delta t$
   from the *fastest* rate constant in your model, and remember that a stable-looking
   answer can still be a few percent wrong.

7. **`solve_ivp` for production, hand-rolled loops for understanding.** Once you can
   write $d\mathbf{y}/dt$ as a function, adaptive higher-order solvers do the rest — and
   for stiff systems (widely separated rate constants) they do it far better than any
   fixed-step scheme.

### Further reading

- Press, Teukolsky, Vetterling & Flannery (2007). *Numerical Recipes*, 3rd ed.,
  chapter 17 — integration of ordinary differential equations, including why you should
  usually not use Euler for real work.
- Hodgkin & Huxley (1952). A quantitative description of membrane current and its
  application to conduction and excitation in nerve. *Journal of Physiology* **117**,
  500–544.
- Rieke & Baylor (1998). Origin of reproducibility in the responses of retinal rods to
  single photons. *Biophysical Journal* **75**, 1836–1857.
- Pugh & Lamb (1993). Amplification and kinetics of the activation steps in
  phototransduction. *Biochimica et Biophysica Acta* **1141**, 111–149.
- Robinson (1964). The mechanics of human saccadic eye movement. *Journal of Physiology*
  **174**, 245–264 — the origin of the linear oculomotor plant model of Part X.
- Dayan & Abbott (2001). *Theoretical Neuroscience*, appendix on numerical integration
  and chapters 5–6 on model neurons.
- The SciPy documentation for `scipy.integrate.solve_ivp`, particularly the discussion of
  stiff versus non-stiff methods.